## 1. Importing Libraries
**What is done in this cell:**
We import Pandas and NumPy for data manipulation, and we bring in `StandardScaler` from Scikit-Learn to handle our feature scaling.



In [19]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

## 2. Load Raw Data
**What is done in this cell:**
We load the raw CSV file and immediately convert the `date` column into a proper Datetime format, setting it as the index.

**Output Explanation:**
Prints the initial shape of our raw dataset so we can track how it changes after preprocessing.

In [20]:
df = pd.read_csv("C:/Users/lalit/Desktop/Projects/Project AQI/aqi-ahmedabad/data/air_quality_historical.csv")
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)

print("Initial Raw Data Shape:", df.shape)
display(df.head(3))

Initial Raw Data Shape: (1298, 11)


,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,aerosol_optical_depth,dust,uv_index,us_aqi,european_aqi
date,,,,,,,,,,,
2022-08-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-08-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-08-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Handling Missing Values
**What is done in this cell:**
Machine Learning models crash when they encounter `NaN` (missing) values. Since this is time-series data, the most logical way to fill missing values is by carrying the previous/next day's data forward/backward (imputation). We use backward fill first (to handle missing data at the very start), followed by a forward fill.

**Output Explanation:**
We print a confirmation checking if there are any remaining null values across the entire dataframe. A result of `0` means the dataset is fully clean of blanks.

In [21]:
df.bfill(inplace=True)
df.ffill(inplace=True)

print("Total missing values after imputation:", df.isnull().sum().sum())

Total missing values after imputation: 0


## 4. Feature Engineering
**What is done in this cell:**
We extract powerful contextual features from the `date` index. Specifically, we extract the `year`, `month`, `day`, and `dayofweek`. We also create binary flags like `is_weekend`, and map the months to Indian `season`s. 

**Output Explanation:**
This enriches our dataset by giving the future ML models human-like context (e.g., "Is this pollution spike happening because it's winter?" or "Is pollution lower because it's the weekend?").

In [22]:
df['year'] = df.index.year
df['month'] = df.index.month
df['day'] = df.index.day
df['dayofweek'] = df.index.dayofweek
df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)  # 5 = Sat, 6 = Sun

def get_season(month):
    if month in [12, 1, 2]: return 'Winter'
    elif month in [3, 4, 5]: return 'Summer'
    elif month in [6, 7, 8, 9]: return 'Monsoon'
    else: return 'Post-Monsoon'

df['season'] = df['month'].apply(get_season)

print("Newly engineered features:", ['year', 'month', 'day', 'dayofweek', 'is_weekend', 'season'])

Newly engineered features: ['year', 'month', 'day', 'dayofweek', 'is_weekend', 'season']


## 5. Encoding Categorical Variables
**What is done in this cell:**
Machine Learning algorithms only understand numbers. The `season` column is text ("Winter", "Summer", etc.). We use **One-Hot Encoding** (`pd.get_dummies`) to convert these categories into binary columns (1s and 0s).

**Output Explanation:**
The text `season` column is dropped, and replaced by 4 new numerical columns like `season_Winter`, `season_Summer`, etc.

In [23]:
df = pd.get_dummies(df, columns=['season'], dtype=int)
display(df.head(2))

,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,aerosol_optical_depth,dust,uv_index,us_aqi,european_aqi,year,month,day,dayofweek,is_weekend,season_Monsoon,season_Post-Monsoon,season_Summer,season_Winter
date,,,,,,,,,,,,,,,,,,,,
2022-08-01,36.726316,22.131579,303.157895,22.810526,10.163158,59.473684,0.498947,11.052632,1.594737,70.947368,47.421053,2022,8,1,0,0,1,0,0,0
2022-08-02,36.726316,22.131579,303.157895,22.810526,10.163158,59.473684,0.498947,11.052632,1.594737,70.947368,47.421053,2022,8,2,1,0,1,0,0,0


## 6. Removing Redundant Data
**What is done in this cell:**
We drop the `european_aqi` column. Since our target variable is `us_aqi`, leaving another AQI column in the dataset causes "data leakage" (the model would perfectly "cheat" by looking at the European index to guess the US index, making the model useless for real-world independent predictions).

**Output Explanation:**
The dataset size is slightly reduced, making it strictly focused on raw pollutants predicting the target.

In [24]:
if 'european_aqi' in df.columns:
    df.drop(columns=['european_aqi'], inplace=True)

print("Columns successfully dropped.")

Columns successfully dropped.


## 7. Feature Scaling
**What is done in this cell:**
Pollutants have vastly different scales (e.g., CO is measured in hundreds, while UV Index is 0-10). Models like Neural Networks or SVMs struggle when features operate on different scales. We use `StandardScaler` to normalize the pollutants so they all have a mean of 0 and a standard deviation of 1.

*(Note: We deliberately DO NOT scale our target variable `us_aqi`, nor our datetime/binary engineered columns).* 

**Output Explanation:**
You will see the pollutant values transform from large raw numbers into small decimal values centered around zero.

In [25]:
features_to_scale = ['pm10', 'pm2_5', 'carbon_monoxide', 'nitrogen_dioxide', 'sulphur_dioxide', 
                     'ozone', 'aerosol_optical_depth', 'dust', 'uv_index']

scaler = StandardScaler()
df[features_to_scale] = scaler.fit_transform(df[features_to_scale])

print("Features successfully scaled.")
display(df[features_to_scale].head(3))

Features successfully scaled.


,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,aerosol_optical_depth,dust,uv_index
date,,,,,,,,,
2022-08-01,-0.576008,-0.524012,-0.441318,0.209593,-0.143563,-0.539947,0.496881,-0.398077,0.00389
2022-08-02,-0.576008,-0.524012,-0.441318,0.209593,-0.143563,-0.539947,0.496881,-0.398077,0.00389
2022-08-03,-0.576008,-0.524012,-0.441318,0.209593,-0.143563,-0.539947,0.496881,-0.398077,0.00389


## 8. Save Preprocessed Data
**What is done in this cell:**
The dataset is fully cleaned, engineered, encoded, and scaled. We now export this finalized dataframe into a new CSV file called `clean_data.csv`.

**Output Explanation:**
A physical file is created in your directory. Future notebooks (like ML modeling) will load this clean file directly instead of doing the preprocessing all over again.

In [27]:
df.to_csv("C:/Users/lalit/Desktop/Projects/Project AQI/aqi-ahmedabad/data/clean_data.csv")
print("Final shape of the clean dataset:", df.shape)
print("\nSuccess! Cleaned data saved to 'C:/Users/lalit/Desktop/Projects/Project AQI/aqi-ahmedabad/data/clean_data.csv'")

Final shape of the clean dataset: (1298, 19)

Success! Cleaned data saved to 'C:/Users/lalit/Desktop/Projects/Project AQI/aqi-ahmedabad/data/clean_data.csv'
